# 2B location-conditioning ablations at 1000 steps

Nine repaired-placement final adapters, nine correct-coordinate evaluations and two shuffled-coordinate evaluations on the fixed BigEarthNet.txt `bench` population.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

evaluation_root = repo_root / "outputs/evaluation"
finetuning_root = repo_root / "outputs/finetuning"
pd.options.display.max_columns = None

primary_manifest = pd.DataFrame([
    {"evaluation_job": "11401", "training_job": "11383", "label": "no_loc", "condition": "no_loc", "location_format": "none", "satclip_l": None, "tokens": 0, "projection_lr_x": None},
    {"evaluation_job": "11402", "training_job": "11384", "label": "loc_text 2dp", "condition": "loc_text", "location_format": "2 decimals", "satclip_l": None, "tokens": 0, "projection_lr_x": None},
    {"evaluation_job": "11409", "training_job": "11400", "label": "loc_text integer", "condition": "loc_text", "location_format": "integer", "satclip_l": None, "tokens": 0, "projection_lr_x": None},
    {"evaluation_job": "11403", "training_job": "11385", "label": "L10 8t 5x", "condition": "loc_embed", "location_format": "embedding", "satclip_l": 10, "tokens": 8, "projection_lr_x": 5},
    {"evaluation_job": "11404", "training_job": "11387", "label": "L10 4t 5x", "condition": "loc_embed", "location_format": "embedding", "satclip_l": 10, "tokens": 4, "projection_lr_x": 5},
    {"evaluation_job": "11405", "training_job": "11388", "label": "L10 8t 2x", "condition": "loc_embed", "location_format": "embedding", "satclip_l": 10, "tokens": 8, "projection_lr_x": 2},
    {"evaluation_job": "11406", "training_job": "11389", "label": "L10 4t 2x", "condition": "loc_embed", "location_format": "embedding", "satclip_l": 10, "tokens": 4, "projection_lr_x": 2},
    {"evaluation_job": "11410", "training_job": "11398", "label": "L40 8t 5x", "condition": "loc_embed", "location_format": "embedding", "satclip_l": 40, "tokens": 8, "projection_lr_x": 5},
    {"evaluation_job": "11411", "training_job": "11399", "label": "L40 4t 2x", "condition": "loc_embed", "location_format": "embedding", "satclip_l": 40, "tokens": 4, "projection_lr_x": 2},
])
primary_manifest["coordinate_setting"] = "correct"
counterfactual_manifest = pd.DataFrame([
    {"evaluation_job": "11420", "training_job": "11400", "label": "loc_text integer [shuffled]", "condition": "loc_text", "location_format": "integer", "satclip_l": None, "tokens": 0, "projection_lr_x": None, "coordinate_setting": "shuffled"},
    {"evaluation_job": "11421", "training_job": "11398", "label": "L40 8t 5x [shuffled]", "condition": "loc_embed", "location_format": "embedding", "satclip_l": 40, "tokens": 8, "projection_lr_x": 5, "coordinate_setting": "shuffled"},
])
evaluation_inventory = pd.concat([primary_manifest, counterfactual_manifest], ignore_index=True)
evaluation_inventory.set_index("evaluation_job")

In [ ]:
def scored_path(job, filename):
    return evaluation_root / str(job) / "scored_predictions" / filename

required = []
for job in evaluation_inventory["evaluation_job"]:
    required.extend([
        evaluation_root / job / "predictions.jsonl",
        scored_path(job, "summary.json"),
        scored_path(job, "sample_scores.jsonl"),
    ])
missing = [str(path.relative_to(repo_root)) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing synced artifacts:\n" + "\n".join(missing))

all_summaries = {
    job: json.loads(scored_path(job, "summary.json").read_text(encoding="utf-8"))
    for job in evaluation_inventory["evaluation_job"]
}
summaries = {job: all_summaries[job] for job in primary_manifest["evaluation_job"]}
print(f"Loaded {len(all_summaries)} scored evaluation runs: {len(primary_manifest)} correct + {len(counterfactual_manifest)} shuffled.")

## Population integrity

In [ ]:
def load_sample_scores(job):
    return pd.read_json(scored_path(job, "sample_scores.jsonl"), lines=True)

sample_scores = {job: load_sample_scores(job) for job in evaluation_inventory["evaluation_job"]}
reference_job = primary_manifest.iloc[0]["evaluation_job"]
reference = sample_scores[reference_job].sort_values("sample_id").reset_index(drop=True)
integrity_rows = []
for job, frame in sample_scores.items():
    ordered = frame.sort_values("sample_id").reset_index(drop=True)
    same_ids = ordered["sample_id"].equals(reference["sample_id"])
    same_metadata = all(
        ordered[column].equals(reference[column])
        for column in ["patch_id", "task_type", "task_category", "split"]
    )
    integrity_rows.append({
        "evaluation_job": job,
        "rows": len(frame),
        "unique_sample_ids": frame["sample_id"].nunique(),
        "same_ids": same_ids,
        "same_metadata": same_metadata,
    })
integrity = evaluation_inventory[["evaluation_job", "label", "coordinate_setting"]].merge(pd.DataFrame(integrity_rows), on="evaluation_job")
assert integrity["same_ids"].all() and integrity["same_metadata"].all()
integrity

## Primary correct-coordinate metrics (9 evaluations)

In [ ]:
def headline_metrics(summary):
    by_type = {row["task_type"]: row for row in summary["by_task_type"]}
    return {
        "BLEU-4": summary["captioning"]["bleu4"],
        "METEOR": summary["captioning"]["meteor"],
        "CIDEr": summary["captioning"]["cider"],
        "Binary accuracy": by_type["binary"]["accuracy"],
        "MCQ accuracy": by_type["mcq"]["accuracy"],
        "BBox mIoU": by_type["bounding box"]["miou"],
    }

headline = primary_manifest[["evaluation_job", "label", "condition"]].copy()
headline = headline.join(pd.DataFrame([headline_metrics(summaries[job]) for job in headline["evaluation_job"]]))
primary = ["BLEU-4", "Binary accuracy", "MCQ accuracy", "BBox mIoU"]
metric_columns = list(headline_metrics(next(iter(summaries.values()))))
headline["Rank"] = headline[primary].rank(ascending=False).mean(axis=1)
headline_format = {metric: "{:.4f}" for metric in metric_columns}
headline_format["Rank"] = "{:.2f}"
display(
    headline.set_index("label")
    .drop(columns=["evaluation_job", "condition"])
    .style.format(headline_format)
    .highlight_max(subset=metric_columns, props="font-weight: bold")
    .highlight_min(subset=["Rank"], props="font-weight: bold")
)

In [ ]:
baseline = headline.set_index("label").loc["no_loc", primary]
headline_deltas = headline.set_index("label")[primary].subtract(baseline)
headline_deltas.style.format("{:+.4f}").background_gradient(cmap="RdYlGn", axis=None, vmin=-0.03, vmax=0.03).set_caption("Delta versus no_loc")

## Task-category detail and direct-geography MCQ

In [ ]:
category_rows = []
for row in primary_manifest.itertuples(index=False):
    for score in summaries[row.evaluation_job]["by_task_category"]:
        category_rows.append({"label": row.label, "evaluation_job": row.evaluation_job, **score})
category_scores = pd.DataFrame(category_rows)

def category_table(task_type, metric):
    table = (
        category_scores[category_scores["task_type"] == task_type]
        .pivot(index="label", columns="task_category", values=metric)
        .reindex(primary_manifest["label"])
    )
    table.index.name = None
    table.columns.name = None
    return table

mcq = category_table("mcq", "accuracy")
display(mcq.style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("MCQ accuracy by category"))
display(mcq[["climate zone", "country", "season"]].style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("Direct-geography MCQ accuracy"))

In [ ]:
display(category_table("binary", "accuracy").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("Binary accuracy by category"))
display(category_table("bounding box", "miou").style.format("{:.3f}").highlight_max(axis=0, props="font-weight: bold").set_caption("Bounding-box mIoU by category"))

## Matched ablation deltas

In [ ]:
comparisons = [
    ("Integer minus 2dp text", "loc_text integer", "loc_text 2dp"),
    ("4t minus 8t at L10/5x", "L10 4t 5x", "L10 8t 5x"),
    ("4t minus 8t at L10/2x", "L10 4t 2x", "L10 8t 2x"),
    ("2x minus 5x at L10/8t", "L10 8t 2x", "L10 8t 5x"),
    ("2x minus 5x at L10/4t", "L10 4t 2x", "L10 4t 5x"),
    ("L40 minus L10 at 8t/5x", "L40 8t 5x", "L10 8t 5x"),
    ("L40 minus L10 at 4t/2x", "L40 4t 2x", "L10 4t 2x"),
]
headline_indexed = headline.set_index("label")
ablation_deltas = pd.DataFrame([
    {"Comparison": name, **(headline_indexed.loc[left, primary] - headline_indexed.loc[right, primary]).to_dict()}
    for name, left, right in comparisons
]).set_index("Comparison")
display(ablation_deltas.style.format("{:+.4f}").background_gradient(cmap="RdYlGn", axis=None, vmin=-0.03, vmax=0.03))

direct_geography = ["climate zone", "country", "season"]
geography_deltas = pd.DataFrame([
    {"Comparison": name, **(mcq.loc[left, direct_geography] - mcq.loc[right, direct_geography]).to_dict()}
    for name, left, right in comparisons
]).set_index("Comparison")
geography_deltas.style.format("{:+.3f}").background_gradient(cmap="RdYlGn", axis=None, vmin=-0.10, vmax=0.10)

## Selected full-run settings

In [ ]:
selection_pairs = [
    ("Coordinate text", "integer", "loc_text integer", "loc_text 2dp"),
    ("Location tokens", "8", "L10 8t 5x", "L10 4t 5x"),
    ("Projection LR", "5x", "L10 8t 5x", "L10 8t 2x"),
    ("SatCLIP L", "40", "L40 8t 5x", "L10 8t 5x"),
]
selection_rows = []
def win_tie_loss(delta, tolerance=1e-12):
    ties = delta.abs() <= tolerance
    wins = (delta > tolerance).sum()
    losses = (delta < -tolerance).sum()
    return f"{wins}/{ties.sum()}/{losses}"

for factor, selected, left, right in selection_pairs:
    primary_delta = headline_indexed.loc[left, primary] - headline_indexed.loc[right, primary]
    geography_delta = mcq.loc[left, direct_geography] - mcq.loc[right, direct_geography]
    selection_rows.append({
        "Factor": factor,
        "Selected": selected,
        "Primary W/T/L": win_tie_loss(primary_delta),
        "Direct geography W/T/L": win_tie_loss(geography_delta),
    })
pd.DataFrame(selection_rows).set_index("Factor")

## Exact prediction disagreement

In [ ]:
def load_predictions(job):
    rows = []
    path = evaluation_root / job / "predictions.jsonl"
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            row = json.loads(line)
            rows.append({key: row.get(key) for key in ["sample_id", "patch_id", "task_type", "task_category", "prediction"]})
    return pd.DataFrame(rows).set_index("sample_id").sort_index()

predictions = {row.label: load_predictions(row.evaluation_job) for row in primary_manifest.itertuples(index=False)}
disagreement_rows = []
for name, left, right in comparisons:
    left_frame, right_frame = predictions[left], predictions[right]
    changed = left_frame["prediction"] != right_frame["prediction"]
    disagreement_rows.append({"Comparison": name, "Task type": "all", "Rows": len(changed), "Changed": int(changed.sum()), "Changed fraction": changed.mean()})
    for task_type, indices in left_frame.groupby("task_type").groups.items():
        subset = changed.loc[indices]
        disagreement_rows.append({"Comparison": name, "Task type": task_type, "Rows": len(subset), "Changed": int(subset.sum()), "Changed fraction": subset.mean()})
disagreement = pd.DataFrame(disagreement_rows)
disagreement.pivot(index="Comparison", columns="Task type", values="Changed fraction").style.format("{:.1%}").set_caption("Exact generated-text disagreement")

## Selected adapters: correct and shuffled coordinates (4 evaluations)

In [ ]:
counterfactual_pairs = pd.DataFrame([
    {"label": "loc_text integer", "correct_job": "11409", "shuffled_job": "11420"},
    {"label": "L40 8t 5x", "correct_job": "11410", "shuffled_job": "11421"},
])
counterfactual_scores = []
counterfactual_deltas = []
counterfactual_disagreement = []
for row in counterfactual_pairs.itertuples(index=False):
    correct_metrics = pd.Series(headline_metrics(all_summaries[row.correct_job]))
    shuffled_metrics = pd.Series(headline_metrics(all_summaries[row.shuffled_job]))
    for setting, job, metrics in [
        ("correct", row.correct_job, correct_metrics),
        ("shuffled", row.shuffled_job, shuffled_metrics),
    ]:
        counterfactual_scores.append({"Condition": row.label, "Coordinate setting": setting, "Evaluation job": job, **metrics.to_dict()})
    counterfactual_deltas.append({"Condition": row.label, **(shuffled_metrics[primary] - correct_metrics[primary]).to_dict()})
    correct_predictions = load_predictions(row.correct_job)
    shuffled_predictions = load_predictions(row.shuffled_job)
    changed = correct_predictions["prediction"] != shuffled_predictions["prediction"]
    counterfactual_disagreement.append({"Condition": row.label, "Changed": int(changed.sum()), "Changed fraction": changed.mean()})

display(pd.DataFrame(counterfactual_scores).set_index(["Condition", "Coordinate setting"]).style.format({metric: "{:.4f}" for metric in metric_columns}).set_caption("Correct and shuffled evaluation scores"))
display(pd.DataFrame(counterfactual_deltas).set_index("Condition").style.format("{:+.4f}").set_caption("Shuffled minus correct"))
display(pd.DataFrame(counterfactual_disagreement).set_index("Condition").style.format({"Changed fraction": "{:.1%}"}))

## Validation trajectories (diagnostic only)

In [ ]:
curve_rows = []
for row in primary_manifest.itertuples(index=False):
    event_files = sorted((finetuning_root / row.training_job / "lightning_logs").glob("version_*/events.out.tfevents.*"))
    if not event_files:
        raise FileNotFoundError(f"Missing TensorBoard event file for training job {row.training_job}")
    accumulator = EventAccumulator(str(event_files[-1]), size_guidance={"scalars": 0})
    accumulator.Reload()
    curve_rows.extend({"label": row.label, "step": event.step + 1, "val_loss": event.value} for event in accumulator.Scalars("val/loss"))
curves = pd.DataFrame(curve_rows)

fig, ax = plt.subplots(figsize=(10, 6))
for label, group in curves.groupby("label", sort=False):
    ax.plot(group["step"], group["val_loss"], marker="o", linewidth=1.6, markersize=3, label=label)
ax.set(xlabel="Optimizer step", ylabel="Teacher-forced validation loss", title="Repaired-placement 1000-step trajectories")
ax.grid(alpha=0.25)
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
plt.show()

final_validation = curves.sort_values("step").groupby("label", sort=False).tail(1).set_index("label")[["step", "val_loss"]]
final_validation.style.format({"step": "{:.0f}", "val_loss": "{:.6f}"}).highlight_min(subset=["val_loss"], props="font-weight: bold")

## Reading notes

- Metrics remain separate; no aggregate rank is used.
- Ablation deltas are matched single-seed point estimates.
- Validation loss is diagnostic only.
- The `bench` results informed configuration selection.